In [1]:
import llm_transparency_tool
from llm_transparency_tool.models.tlens_model import TransformerLensTransparentLlm
from llm_transparency_tool.routes.graph import (
    build_full_graph,
    build_paths_to_predictions,
)

ModuleNotFoundError: No module named 'streamlit'

In [ ]:
import torch
import pandas as pd
from llm_transparency_tool.models.tlens_model import TransformerLensTransparentLlm


sentence = "When Mary and John went to the store, John gave a drink to"
model_name = "meta-llama/Llama-3.2-1B"

try:
    model = TransformerLensTransparentLlm(model_name=model_name, device="cpu")
    print(f"Successfully loaded model: {model_name}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure you have an internet connection and the model name is correct.")
    raise

model.run([sentence])
print("Inference complete.")

def print_top_tokens(vector, title, n_top=10, n_bottom=5):
    logits = model.unembed(vector, normalize=True)
    top_scores, top_indices = torch.topk(logits, n_top)
    bottom_scores, bottom_indices = torch.topk(logits, n_bottom, largest=False)
    top_tokens = model.tokens_to_strings(top_indices)
    bottom_tokens = model.tokens_to_strings(bottom_indices)
    
    print(f"\n{'---'*5} {title} {'---'*5}")
    df_top = pd.DataFrame({"Token": top_tokens, "Score": [f"{s:.4f}" for s in top_scores.tolist()]})
    print("Top Tokens (Promoted):\n" + df_top.to_string(index=False))
    print("-" * 20)
    df_bottom = pd.DataFrame({"Token": bottom_tokens, "Score": [f"{s:.4f}" for s in bottom_scores.tolist()]})
    print("Bottom Tokens (Suppressed):\n" + df_bottom.to_string(index=False))
    print(f"{'---'*15}\n")

layer_to_analyze = 11
token_idx_to_analyze = -1
batch_idx = 0


single_token_scalar = model.tokens()[batch_idx, token_idx_to_analyze]
analyzed_token_str = model.tokens_to_strings(torch.tensor([single_token_scalar]))[0]

print(f"\nStarting analysis for token '{analyzed_token_str}' at Layer {layer_to_analyze}...")

vec_after_attn = model.residual_after_attn(layer_to_analyze)[batch_idx, token_idx_to_analyze]
attn_output_vector = model.attention_output(batch_idx, layer_to_analyze, token_idx_to_analyze)

print_top_tokens(vec_after_attn, f"State of Residual Stream (L{layer_to_analyze} After-Attn)")
print_top_tokens(attn_output_vector, f"Change from Attention Block (L{layer_to_analyze})")